# Lecture 11 — Lab: Train a Sequence Model (char-level Shakespeare)

In the sandbox you built a tiny next-char RNN by hand and watched two toy knobs:
the **masked loss** (`ignore_index` so `<pad>` never counts) and the **sampling
temperature** that reshaped `softmax(logits / T)` on `"abcabc…"`. That was the
mechanism on toy numbers — **now train the real thing**: a character-level
language model on ~1 MB of **tiny-shakespeare**, on a GPU, whose payoff is
*generating* new Shakespeare-ish text.

Same five-step training loop you already know; the new wrinkles are a real char
vocabulary, fixed-length `(sequence → next-char)` batches, and an
**autoregressive sampling loop** that feeds the model its own output to write
text.

> **Runtime → Change runtime type → T4 GPU** before you start. The whole notebook
> is sized to finish in **~2–3 minutes** on a free T4.

## 1. Setup & GPU check

Imports, a fixed seed for reproducibility, and the device. If this prints `cpu`,
go to **Runtime → Change runtime type → T4 GPU** and re-run — it will still work
on CPU but slower. We also define the two small helpers each notebook carries on
its own: `plot_curves` and a next-char `accuracy`.

In [ ]:
import math
import urllib.request

import matplotlib.pyplot as plt
import torch
from torch import nn

torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU "
          "for the intended ~2-3 min run (it still works on CPU, just slower).")


def plot_curves(history):
    """Plot train/val loss per epoch from a dict of lists."""
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(6, 4))
    plt.plot(epochs, history["train_loss"], "o-", label="train loss")
    plt.plot(epochs, history["val_loss"], "o-", label="val loss")
    plt.xlabel("epoch")
    plt.ylabel("cross-entropy (nats)")
    plt.title("Training curve")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


@torch.no_grad()
def accuracy(model, x, y, batch_size=512):
    """Next-char top-1 accuracy over (x, y) id tensors, batched to fit memory."""
    model.eval()
    correct = total = 0
    for i in range(0, len(x), batch_size):
        xb = x[i:i + batch_size].to(device)
        yb = y[i:i + batch_size].to(device)
        preds = model(xb).argmax(dim=-1)        # (batch, time)
        correct += (preds == yb).sum().item()
        total += yb.numel()
    return correct / total

## 2. Get the real data — tiny-shakespeare

We download Karpathy's **tiny-shakespeare** corpus (~1 MB of the plays,
concatenated as plain text) at runtime — Colab has internet. There is no fixed
"label" here: a language model's supervision comes *for free* from the text
itself — every character's target is simply **the next character**.

What's real/messy about it: it's raw play text — mixed case, punctuation,
newlines, character names in caps, blank lines. We do **not** clean it; the model
learns the vocabulary as-is. We split the single long string into train / val /
test by **contiguous slices** (90% / 5% / 5%) so the validation and test text is
held out from training.

In [ ]:
URL = ("https://raw.githubusercontent.com/karpathy/char-rnn/"
       "master/data/tinyshakespeare/input.txt")
text = urllib.request.urlopen(URL).read().decode("utf-8")

# Character vocabulary: stoi / itos. Every distinct character is a token.
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}


def encode(s):
    return [stoi[c] for c in s]


def decode(ids):
    return "".join(itos[int(i)] for i in ids)


# Encode the whole corpus once, then split by contiguous slices.
data = torch.tensor(encode(text), dtype=torch.long)
n = len(data)
n_train = int(0.90 * n)
n_val = int(0.95 * n)
train_data = data[:n_train]
val_data = data[n_train:n_val]
test_data = data[n_val:]

print(f"corpus chars : {n:,}")
print(f"vocab size V : {vocab_size}")
print(f"train / val / test : {len(train_data):,} / {len(val_data):,} / {len(test_data):,}")

## 3. Look at the data

Seeing the corpus *is* the real-dataset experience. We print a raw excerpt, the
full character vocabulary, and one concrete `(input → target)` next-char pair so
you can eyeball the **shift-by-one** alignment that defines the task.

In [ ]:
print("----- raw excerpt (first 250 chars) -----")
print(text[:250])

print("\n----- vocabulary (", vocab_size, "chars) -----", sep="")
# unicode_escape so spaces / newlines are visible
print("".join(chars).encode("unicode_escape").decode())

# One (input, target) pair: target is the input shifted left by one.
demo = train_data[:12]
print("\n----- next-char alignment (shift by one) -----")
print("input  ids:", demo[:-1].tolist())
print("target ids:", demo[1:].tolist())
print("input  txt:", repr(decode(demo[:-1])))
print("target txt:", repr(decode(demo[1:])))

## 4. Build the model

A character-level language model is the same three-layer stack from the lecture:
`nn.Embedding → nn.LSTM → nn.Linear(vocab)`. The embedding turns each char id
into a learned vector, the LSTM carries context across the sequence, and the
linear head predicts a distribution over the **next** character at every step.

This is the cell that mirrors what you built from scratch — read it (and try
writing it yourself) before running.

In [ ]:
# 🔧 Your turn — this mirrors what you built from scratch; read it (and try
# writing it yourself) before running.
class CharLM(nn.Module):
    def __init__(self, vocab_size, emb=64, hidden=256, num_layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb)
        self.lstm = nn.LSTM(emb, hidden, num_layers=num_layers, batch_first=True)
        self.head = nn.Linear(hidden, vocab_size)   # logits over the next char

    def forward(self, x, state=None):
        # x: (batch, time) of char ids; returns logits (batch, time, vocab).
        # `state` lets the autoregressive sampler carry the LSTM hidden/cell
        # state across calls; when None the LSTM starts from zeros.
        e = self.embed(x)                 # (batch, time, emb)
        out, state = self.lstm(e, state)  # out: (batch, time, hidden)
        logits = self.head(out)           # (batch, time, vocab)
        self._state = state               # stash for the sampler (see generate)
        return logits


model = CharLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"trainable params: {n_params:,}")

## 5. Generate text — the autoregressive sampling loop

**This is the payoff of the whole lab.** A trained next-char model writes text by
*rolling out*: start from a seed string, get the logits for the next char, divide
them by a **temperature** `T`, softmax to a probability, **sample** one char
(`torch.multinomial`), append it, and feed it back in — repeat.

- **Low `T` (e.g. 0.5)** sharpens toward the top choice → safe but *repetitive*.
- **`T = 1.0`** samples from the model's own distribution.
- **High `T` (e.g. 1.5)** flattens the distribution → diverse but *gibberish*.

(`T → 0` is greedy `argmax`; we clamp `T` to a small floor so we never divide by
zero.)

In [ ]:
# 🔧 Your turn — the autoregressive sampling loop with temperature.
# This is the motivating payoff: feed the model its own output to GENERATE text.
@torch.no_grad()
def generate(model, seed="ROMEO:", n_new=300, temperature=1.0):
    model.eval()
    T = max(temperature, 1e-2)            # guard against divide-by-zero
    ids = torch.tensor([encode(seed)], dtype=torch.long, device=device)  # (1, len)

    # Warm up the LSTM state on the seed; model.forward stashes state in _state.
    logits = model(ids)
    state = model._state
    out_ids = []
    for _ in range(n_new):
        last_logits = logits[:, -1, :] / T            # (1, vocab)
        probs = torch.softmax(last_logits, dim=-1)    # (1, vocab)
        next_id = torch.multinomial(probs, num_samples=1)  # (1, 1) — SAMPLE, not argmax
        out_ids.append(int(next_id))
        logits = model(next_id, state)                # feed the char back in...
        state = model._state                          # ...carrying the LSTM state
    return seed + decode(out_ids)


# Before training the model is random, so this is gibberish — that's expected.
print(generate(model, seed="ROMEO:", n_new=200, temperature=1.0))

## 6. Train

The familiar five-step loop — forward, loss, `zero_grad`, `backward`, `step` —
with three sequence-specific touches:

1. **Random fixed-length blocks.** We sample `(block, next-char-block)` pairs of
   length `BLOCK` from the corpus on the fly; the target is the input shifted by
   one. (No `<pad>` here — every block is full length — so no `ignore_index`
   needed; padding/masking was the sandbox toy.)
2. **Flatten time into batch** before cross-entropy: logits `(B*T, vocab)` vs
   targets `(B*T,)`.
3. **Gradient clipping** (`clip_grad_norm_(..., 1.0)`) — cheap insurance against
   the exploding gradients recurrent models are prone to.

We print a **sample every epoch** so you can watch the text go from noise to
something Shakespeare-ish. Sized for ~2–3 min on a T4: small model, `BLOCK=128`,
a few hundred steps per epoch.

In [ ]:
BLOCK = 128          # sequence length per training example
BATCH = 64           # sequences per batch
STEPS_PER_EPOCH = 200
EPOCHS = 4
LR = 3e-3


def get_batch(split_data, batch=BATCH, block=BLOCK):
    """Sample `batch` random length-`block` (x, y) pairs; y is x shifted by one."""
    ix = torch.randint(0, len(split_data) - block - 1, (batch,))
    x = torch.stack([split_data[i:i + block] for i in ix])
    y = torch.stack([split_data[i + 1:i + 1 + block] for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def eval_loss(split_data, n_batches=20):
    """Average cross-entropy over a few random batches of a split."""
    model.eval()
    losses = []
    for _ in range(n_batches):
        x, y = get_batch(split_data)
        logits = model(x)
        loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
        losses.append(loss.item())
    return sum(losses) / len(losses)


optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for _ in range(STEPS_PER_EPOCH):
        x, y = get_batch(train_data)                       # (B, T) ids
        logits = model(x)                                  # (B, T, vocab)
        loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # exploding-grad insurance
        optimizer.step()
        running += loss.item()

    train_loss = running / STEPS_PER_EPOCH
    val_loss = eval_loss(val_data)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"epoch {epoch}: train {train_loss:.3f} | val {val_loss:.3f}")
    print("  sample:", repr(generate(model, seed="ROMEO:", n_new=120, temperature=0.8)))
    print()

### 🔧 Diagnose & fix the broken run

This cell is **isolated** — it does not touch the model you just trained. It
trains a *fresh* model with one planted bug, and the loss barely moves.

**Your task:** run it, watch the loss sit near the random baseline `log(V)`, and
find the bug. *Hint from the lecture's tuning table: a loss stuck near `log V`
usually means the **targets are misaligned**.* Fix the one line, re-run, and
confirm the loss now falls well below `log(V)`.

In [ ]:
# 🔧 BROKEN on purpose — one line is wrong. Find it and fix it.
buggy = CharLM(vocab_size).to(device)
buggy_opt = torch.optim.Adam(buggy.parameters(), lr=LR)
log_V = math.log(vocab_size)
print(f"random-guess baseline  log(V) = {log_V:.3f}")

for step in range(150):
    ix = torch.randint(0, len(train_data) - BLOCK - 1, (BATCH,))
    x = torch.stack([train_data[i:i + BLOCK] for i in ix]).to(device)
    # BUG: the target is identical to the input (no shift) — the model is asked
    # to copy what it was just handed, so it can never beat the log(V) baseline.
    # FIX: shift the target left by one -> train_data[i + 1 : i + 1 + BLOCK]
    y = torch.stack([train_data[i:i + BLOCK] for i in ix]).to(device)
    logits = buggy(x)
    loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
    buggy_opt.zero_grad()
    loss.backward()
    buggy_opt.step()
    if step % 30 == 0:
        print(f"step {step:3d}: loss {loss.item():.3f}")

print("\nLoss stuck near log(V)? Re-read the BUG comment, fix the `y = ...` line, "
      "and re-run — a correct model drops well below the baseline within ~150 steps.")

## 7. Evaluate

Two views of the trained model:

1. **The training curve** (`plot_curves`) — train vs. val loss per epoch.
2. **Perplexity** = `exp(mean cross-entropy)`, the standard language-model
   metric. Read it as *"how many characters the model is effectively choosing
   among at each step."* The random-guess baseline is `exp(log V) = V`; a trained
   model should land **well below `V`**.

In [ ]:
plot_curves(history)

test_ce = eval_loss(test_data, n_batches=40)
log_V = math.log(vocab_size)
xv, yv = get_batch(val_data, batch=256)
print(f"test cross-entropy : {test_ce:.3f} nats")
print(f"test perplexity    : {math.exp(test_ce):.2f}")
print(f"baseline V (= exp(log V)) : {vocab_size}   "
      f"(uniform random over {vocab_size} chars)")
print(f"log(V) loss floor  : {log_V:.3f} nats")
print(f"next-char accuracy (val): {accuracy(model, xv, yv):.3f}")

### Generated samples at three temperatures

The same trained model, sampled at **T = 0.5 / 1.0 / 1.5**. Watch the trade-off:
low `T` is clean but loops/repeats, high `T` drifts into gibberish, `T = 1.0`
sits in between.

In [ ]:
for T in (0.5, 1.0, 1.5):
    print(f"================= temperature = {T} =================")
    print(generate(model, seed="ROMEO:", n_new=300, temperature=T))
    print()

## Experiment — modify & observe

Try these and watch what changes (each is a small edit + re-run):

1. **Temperature sweep on generation.** Call `generate(...)` with `temperature`
   in `[0.2, 0.5, 0.8, 1.0, 1.2, 1.5, 2.0]` from the *same* trained model. Find
   where it tips from *repetitive* (low) to *coherent* to *gibberish* (high).
   **No retraining needed** — temperature is a generation-time knob only.
2. **Train a bit longer.** Bump `EPOCHS` to `8` (and/or `STEPS_PER_EPOCH` to
   `400`), re-run training, and watch the per-epoch samples get more word-like and
   the perplexity drop. (Still well under a few minutes on a T4.)
3. **Beat the baseline → then beat *this*.** Your trained perplexity must beat the
   `V` random baseline (it will, easily). **Target: push test perplexity below
   ~6.5.** Knobs: `hidden` (e.g. 256 → 384), `num_layers=2`, `BLOCK`, `LR`, more
   steps.

Fill in this ablation table from your own runs:

| change | epochs | test perplexity | best-sample quality (1–5) |
|---|---|---|---|
| baseline (as shipped) |  |  |  |
| EPOCHS = 8 |  |  |  |
| hidden = 384 |  |  |  |
| num_layers = 2 |  |  |  |

## Reflect — report YOUR results

Answer from the numbers and samples *you* produced:

1. **Perplexity vs `log(V)`.** What was your final test perplexity, and how far
   below the baseline `V` did it land? Restate it the lecture's way: *how many
   characters is the model effectively choosing among per step?*
2. **What does temperature change — and what does it *not*?** Compare your
   `T = 0.5` vs `T = 1.5` samples. Did the *most-likely* next character change, or
   only how often the model gambles on a less-likely one? (Does temperature affect
   training at all?)
3. **Best vs worst sample.** Paste your favourite generated passage and your worst
   one, and name the temperature each came from.
4. **Train–val gap.** Did val loss track train loss, or start to lag (overfitting)?
   What would you turn first to close a gap — more data, fewer epochs, a smaller
   model, or dropout?